# Homework 6: Batch Processing
**Data Engineering Zoomcamp 2026**

Dataset: Yellow Taxi Trips — November 2025

In [1]:
import os
import urllib.request
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## Question 1 — Spark Version

In [2]:
# Fix for Java 17/21 + Hadoop incompatibility (getSubject not supported)
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--driver-java-options '
    '"-Djava.security.manager=allow '
    '--add-opens=java.base/javax.security.auth=ALL-UNNAMED '
    '--add-opens=java.base/java.lang=ALL-UNNAMED" '
    'pyspark-shell'
)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("de-zoomcamp-hw6")
    .config("spark.driver.extraJavaOptions",
            "-Djava.security.manager=allow "
            "--add-opens=java.base/javax.security.auth=ALL-UNNAMED "
            "--add-opens=java.base/java.lang=ALL-UNNAMED "
            "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED")
    .config("spark.executor.extraJavaOptions",
            "-Djava.security.manager=allow "
            "--add-opens=java.base/javax.security.auth=ALL-UNNAMED")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED
26/03/09 23:00:19 WARN Utils: Your hostname, MacBook-Pro-Alexander.local resolves to a loopback address: 127.0.0.1; using 192.168.31.132 instead (on interface en0)
26/03/09 23:00:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 23:00:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.3


## Download Data

In [3]:
DATA_URL   = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
DATA_FILE  = "../data/yellow_tripdata_2025-11.parquet"
ZONES_URL  = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
ZONES_FILE = "../data/taxi_zone_lookup.csv"

os.makedirs("../data", exist_ok=True)

if not os.path.exists(DATA_FILE):
    print("Downloading yellow_tripdata_2025-11.parquet (~66 MB)...")
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print("Done.")
else:
    print("yellow_tripdata_2025-11.parquet already exists.")

if not os.path.exists(ZONES_FILE):
    print("Downloading taxi_zone_lookup.csv...")
    urllib.request.urlretrieve(ZONES_URL, ZONES_FILE)
    print("Done.")
else:
    print("taxi_zone_lookup.csv already exists.")

yellow_tripdata_2025-11.parquet already exists.
taxi_zone_lookup.csv already exists.


## Question 2 — File Size of Yellow November 2025

In [4]:
file_size_mb = os.path.getsize(DATA_FILE) / (1024 * 1024)
print(f"File size: {file_size_mb:.1f} MB")

File size: 67.8 MB


## Load DataFrame

In [5]:
df = spark.read.parquet(DATA_FILE)
df.printSchema()
print(f"Total rows: {df.count():,}")

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



[Stage 1:>                                                          (0 + 4) / 4]

Total rows: 4,181,444


## Question 3 — Count Records (November 15, 2025)

In [6]:
df_nov15 = df.filter(F.to_date("tpep_pickup_datetime") == "2025-11-15")
count_nov15 = df_nov15.count()
print(f"Records on 2025-11-15: {count_nov15:,}")

[Stage 4:>                                                          (0 + 4) / 4]

Records on 2025-11-15: 162,604


## Question 4 — Longest Trip Duration (hours)

In [7]:
df_dur = df.withColumn(
    "duration_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)
max_dur = df_dur.agg(F.max("duration_hours")).collect()[0][0]
print(f"Longest trip: {max_dur:.1f} hours")

[Stage 7:>                                                          (0 + 4) / 4]

Longest trip: 90.6 hours


## Question 5 — Spark UI Port

In [8]:
print(f"Spark UI URL: {spark.sparkContext.uiWebUrl}")
print("Default port: 4040")

Spark UI URL: http://192.168.31.132:4040
Default port: 4040


## Question 6 — Least Frequent Pickup Location Zone

In [9]:
zones = spark.read.option("header", "true").csv(ZONES_FILE)

pickup_counts = (
    df
    .groupBy("PULocationID")
    .count()
    .withColumnRenamed("PULocationID", "LocationID")
    .withColumnRenamed("count", "pickup_count")
)

result = (
    pickup_counts
    .join(zones, on="LocationID", how="left")
    .orderBy("pickup_count")
    .select("Zone", "pickup_count")
)

print("Least frequent pickup zones:")
result.show(10, truncate=False)

Least frequent pickup zones:


[Stage 12:=============================>                            (2 + 2) / 4]

+---------------------------------------------+------------+
|Zone                                         |pickup_count|
+---------------------------------------------+------------+
|Governor's Island/Ellis Island/Liberty Island|1           |
|Arden Heights                                |1           |
|Eltingville/Annadale/Prince's Bay            |1           |
|Port Richmond                                |3           |
|Rossville/Woodrow                            |4           |
|Rikers Island                                |4           |
|Green-Wood Cemetery                          |4           |
|Great Kills                                  |4           |
|Jamaica Bay                                  |5           |
|Westerleigh                                  |12          |
+---------------------------------------------+------------+
only showing top 10 rows



In [10]:
spark.stop()
print("Done!")

Done!
